In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/Dataset.zip" -d "/content/sample_data"

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 33.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
import os
import cv2
import shutil
import json
import numpy as np
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import albumentations as A
import torch

# Define paths
dataset_path = '/content/sample_data/Dataset'  # Path for Google Colab
image_dir = os.path.join(dataset_path, 'Images')
json_path = os.path.join(dataset_path, 'Annotations.json')

# Check if dataset path exists
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")
if not os.path.exists(image_dir):
    raise FileNotFoundError(f"Images directory does not exist: {image_dir}")
if not os.path.exists(json_path):
    raise FileNotFoundError(f"Annotations file does not exist: {json_path}")

# Clear existing images and labels directories to avoid conflicts
output_base = dataset_path
for split in ['train', 'val', 'test']:
    images_split_dir = os.path.join(output_base, 'images', split)
    labels_split_dir = os.path.join(output_base, 'labels', split)
    if os.path.exists(images_split_dir):
        shutil.rmtree(images_split_dir)
    if os.path.exists(labels_split_dir):
        shutil.rmtree(labels_split_dir)
    os.makedirs(images_split_dir, exist_ok=True)
    os.makedirs(labels_split_dir, exist_ok=True)

# Load COCO annotations
with open(json_path, 'r') as f:
    coco_data = json.load(f)

# Map image IDs to filenames and sizes
image_id_to_filename = {img['id']: img['file_name'] for img in coco_data['images']}
image_id_to_size = {img['id']: (img['width'], img['height']) for img in coco_data['images']}

# Map category IDs to YOLO class IDs (0-based)
category_id_to_yolo = {cat['id']: idx for idx, cat in enumerate(coco_data['categories'])}
class_names = [cat['name'] for cat in coco_data['categories']]

# Verify class names
print("Class names from annotations:", class_names)
assert class_names == ['CreashBarrier', 'KerbPaint', 'MarkingPaint', 'RubberSpeedBreaker'], "Class names do not match expected classes."

# Split images
image_files = [img['file_name'] for img in coco_data['images']]
train_images, temp_images = train_test_split(image_files, test_size=0.2, random_state=42)
val_images, test_images = train_test_split(temp_images, test_size=0.5, random_state=42)

print(f"Train: {len(train_images)} images")
print(f"Validation: {len(val_images)} images")
print(f"Test: {len(test_images)} images")

# Convert COCO to YOLO format and move files
def convert_to_yolo(split, image_list):
    image_split_dir = os.path.join(output_base, 'images', split)
    label_split_dir = os.path.join(output_base, 'labels', split)

    for img_file in image_list:
        # Move image
        src_img = os.path.join(image_dir, img_file)
        dst_img = os.path.join(image_split_dir, img_file)
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
        else:
            print(f"Image not found: {src_img}")
            continue

        # Find image ID
        img_id = next(img['id'] for img in coco_data['images'] if img['file_name'] == img_file)
        img_width, img_height = image_id_to_size[img_id]

        # Get annotations for this image
        annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] == img_id]

        # Convert to YOLO format
        yolo_labels = []
        for ann in annotations:
            class_id = int(category_id_to_yolo[ann['category_id']])
            bbox = ann['bbox']  # [x_min, y_min, width, height]

            # Calculate YOLO coordinates
            x_center = (bbox[0] + bbox[2] / 2) / img_width
            y_center = (bbox[1] + bbox[3] / 2) / img_height
            width = bbox[2] / img_width
            height = bbox[3] / img_height

            # Clamp coordinates to [0.0, 1.0]
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            width = max(0.0, min(1.0, width))
            height = max(0.0, min(1.0, height))

            # Ensure x_max and y_max are in bounds
            x_max = x_center + width / 2
            y_max = y_center + height / 2
            if x_max > 1.0:
                width = (1.0 - x_center) * 2
            if y_max > 1.0:
                height = (1.0 - y_center) * 2
            if x_center - width / 2 < 0:
                width = x_center * 2
            if y_center - height / 2 < 0:
                height = y_center * 2

            # Ensure width and height are at least 0.001 to avoid zero-size boxes
            width = max(0.001, width)
            height = max(0.001, height)

            yolo_labels.append(f"{class_id} {x_center} {y_center} {width} {height}")

        # Generate label filename by removing any '_aug_X' suffix and replacing image extension with .txt
        base_name = os.path.splitext(img_file)[0]  # Get the filename without extension
        # Remove '_aug_X' suffix if present (e.g., 'image_aug_1' -> 'image')
        if '_aug_' in base_name:
            base_name = base_name.split('_aug_')[0]
        label_file = os.path.join(label_split_dir, f"{base_name}.txt")

        # Write YOLO label file
        with open(label_file, 'w') as f:
            f.write('\n'.join(yolo_labels))

# Process each split
convert_to_yolo('train', train_images)
convert_to_yolo('val', val_images)
convert_to_yolo('test', test_images)

# Create data.yaml for YOLOv8
data_yaml = f"""
train: {os.path.join(output_base, 'images', 'train')}
val: {os.path.join(output_base, 'images', 'val')}
test: {os.path.join(output_base, 'images', 'test')}

nc: {len(class_names)}
names: {class_names}
"""

data_yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

print("Dataset split and conversion to YOLO format completed.")
print(f"data.yaml created with class names: {class_names}")

# Define paths
splits = ['train', 'val', 'test']

# Define class names and updated oversampling factors
oversample_factors = {
    'CreashBarrier': 2,
    'KerbPaint': 1,
    'MarkingPaint': 4,
    'RubberSpeedBreaker': 4
}

# Define enhanced augmentation pipeline
transform = A.Compose([
    A.Resize(height=640, width=640),
    A.HorizontalFlip(p=0.8),
    A.RandomRotate90(p=0.8),
    A.Rotate(limit=45, p=0.8),
    A.RandomScale(scale_limit=0.3, p=0.8),
    A.Affine(translate_percent=0.1, scale=(0.8, 1.2), rotate=(-45, 45), p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.8),
    A.GaussNoise(p=0.6),
    A.Blur(blur_limit=5, p=0.6),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.4, p=0.8),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Function to clamp bounding box coordinates
def clamp_bbox(bbox):
    center_x, center_y, width, height = bbox
    center_x = max(0.0, min(1.0, center_x))
    center_y = max(0.0, min(1.0, center_y))
    width = max(0.0, min(1.0, width))
    height = max(0.0, min(1.0, height))

    x_max = center_x + width / 2
    y_max = center_y + height / 2
    if x_max > 1.0:
        width = (1.0 - center_x) * 2
    if y_max > 1.0:
        height = (1.0 - center_y) * 2
    if center_x - width / 2 < 0:
        width = center_x * 2
    if center_y - height / 2 < 0:
        height = center_y * 2

    return [center_x, center_y, width, height]

# Preprocess and augment images with oversampling
def preprocess_images(image_dir, label_dir, output_image_dir, output_label_dir):
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)

    processed_count = 0
    image_files = os.listdir(image_dir)

    # Only process original images (exclude augmented images with '_aug_' in the name)
    image_files = [f for f in image_files if '_aug_' not in f]

    for img_file in image_files:
        img_path = os.path.join(image_dir, img_file)

        # Generate the base name for the label file (without '_aug_X' suffix)
        base_name = os.path.splitext(img_file)[0]
        label_path = os.path.join(label_dir, f"{base_name}.txt")

        # Read image and labels
        img = cv2.imread(img_path)
        if img is None:
            print(f"Skipping corrupted image: {img_file}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        try:
            with open(label_path, 'r') as f:
                labels = [line.strip().split() for line in f.readlines()]
        except Exception as e:
            print(f"Error reading label file {label_path}: {e}")
            continue

        if not labels:
            print(f"No annotations for {img_file}, copying without augmentation.")
            cv2.imwrite(os.path.join(output_image_dir, img_file), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            shutil.copy(label_path, os.path.join(output_label_dir, img_file.replace('.jpg', '.txt')))
            processed_count += 1
            continue

        class_labels = [int(label[0]) for label in labels]
        bboxes = [list(map(float, label[1:])) for label in labels]

        # Clamp bounding box coordinates
        bboxes = [clamp_bbox(bbox) for bbox in bboxes]

        # Determine oversampling factor based on the classes present
        max_factor = 1
        for class_id in class_labels:
            class_name = class_names[class_id]
            factor = oversample_factors[class_name]
            max_factor = max(max_factor, factor)

        # Apply augmentations for each oversampled instance
        for i in range(max_factor):
            try:
                augmented = transform(image=img, bboxes=bboxes, class_labels=class_labels)
                aug_img = augmented['image']
                aug_bboxes = augmented['bboxes']
                aug_labels = augmented['class_labels']
            except Exception as e:
                print(f"Error augmenting {img_file} (iteration {i+1}): {e}")
                continue

            # Save augmented image
            if i == 0:
                aug_img_file = img_file
            else:
                base, ext = os.path.splitext(img_file)
                aug_img_file = f"{base}_aug_{i}{ext}"

            aug_img = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(output_image_dir, aug_img_file), aug_img)

            # Save updated labels
            label_file = aug_img_file.replace('.jpg', '.txt')
            with open(os.path.join(output_label_dir, label_file), 'w') as f:
                for class_id, bbox in zip(aug_labels, aug_bboxes):
                    f.write(f"{class_id} {' '.join(map(str, bbox))}\n")

            processed_count += 1

    print(f"Processed {processed_count} images in {image_dir}")

# Preprocess all splits
for split in splits:
    print(f"\nPreprocessing {split} split...")
    image_dir = os.path.join(dataset_path, 'images', split)
    label_dir = os.path.join(dataset_path, 'labels', split)
    if not os.path.exists(image_dir) or not os.path.exists(label_dir):
        print(f"Directories not found: {image_dir}, {label_dir}")
        continue
    preprocess_images(
        image_dir,
        label_dir,
        image_dir,
        label_dir
    )

print("\nPreprocessing completed.")

# Load the YOLOv8m model with pretrained weights
model = YOLO('yolov8m.pt')  # Updated to use pretrained weights

# Train the model with optimized hyperparameters to aim for 85% mAP@50
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=32,
    name='yolov8m_85_accuracy',
    project='/content/sample_data/Dataset/runs',
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=25,
    save=True,
    val=True,
    exist_ok=True,
    cls=2.0,  # Emphasize classification loss
    box=8.5,  # Improve bounding box precision
    lr0=0.0003,  # Lower learning rate for finer convergence
    cos_lr=True,
    optimizer='AdamW',
    mixup=0.1,  # Add MixUp augmentation
)

# Evaluate the model on the test set
metrics = model.val(split='test')

# Print evaluation metrics
print("\nEvaluation Metrics on Test Set:")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50:95: {metrics.box.map:.4f}")
print("Per-class AP@50:")
for i, ap in enumerate(metrics.box.maps):
    print(f"{model.names[i]}: {ap:.4f}")

# Save the retrained model
model.save('/content/sample_data/Dataset/yolov8m_85_accuracy.pt')

print("Retraining and evaluation completed.")

Class names from annotations: ['CreashBarrier', 'KerbPaint', 'MarkingPaint', 'RubberSpeedBreaker']
Train: 2880 images
Validation: 360 images
Test: 360 images
Dataset split and conversion to YOLO format completed.
data.yaml created with class names: ['CreashBarrier', 'KerbPaint', 'MarkingPaint', 'RubberSpeedBreaker']

Preprocessing train split...
Processed 5051 images in /content/sample_data/Dataset/images/train

Preprocessing val split...
Processed 637 images in /content/sample_data/Dataset/images/val

Preprocessing test split...
Processed 612 images in /content/sample_data/Dataset/images/test

Preprocessing completed.


100%|██████████| 49.7M/49.7M [00:00<00:00, 121MB/s]


Ultralytics 8.3.140 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=8.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sample_data/Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8m_85_accuracy, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=25, perspective=0.0, plots=True, 

100%|██████████| 755k/755k [00:00<00:00, 19.5MB/s]

Overriding model.yaml nc=80 with nc=4

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

...

                   all        723        803       0.85      0.847      0.847      0.606
         CreashBarrier        197        264      0.867      0.799      0.847      0.503
             KerbPaint         87        187      0.698      0.749      0.699      0.454
          MarkingPaint        352        352      0.983      0.994      0.994      0.861
Speed: 0.2ms preprocess, 10.3ms inference, 0.0ms loss, 4.8ms postprocess per image
Results saved to /content/sample_data/Dataset/runs/yolov8m_85_accuracy
Ultralytics 8.3.140 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 22.8±12.3 MB/s, size: 221.2 KB)


val: Scanning /content/sample_data/Dataset/labels/test... 612 images, 86 backgrounds, 0 corrupt: 100%|██████████| 696/696 [00:02<00:00, 315.38it/s]

val: New cache created: /content/sample_data/Dataset/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 22/22 [00:21<00:00,  1.02it/s]


                   all        696        753      0.864      0.827       0.85      0.611
         CreashBarrier        210        261      0.902      0.777      0.864      0.509
             KerbPaint         92        184      0.708      0.717      0.692      0.453
          MarkingPaint        308        308      0.982      0.987      0.994       0.87
Speed: 1.7ms preprocess, 23.3ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/sample_data/Dataset/runs/yolov8m_85_accuracy

Evaluation Metrics on Test Set:
mAP@50: 0.8500
mAP@50:95: 0.6105
Per-class AP@50:
CreashBarrier: 0.5093
KerbPaint: 0.4527
MarkingPaint: 0.8696
RubberSpeedBreaker: 0.6105
Retraining and evaluation completed.


In [ ]:
import os
import cv2
import shutil
import json
import numpy as np
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import albumentations as A
import torch

# Define paths
dataset_path = '/content/sample_data/Dataset'  # Path for Google Colab
image_dir = os.path.join(dataset_path, 'Images')
json_path = os.path.join(dataset_path, 'Annotations.json')
pretrained_model_path = '/content/sample_data/Dataset/runs/yolov8m_85_accuracy/weights/best.pt'  # Path to the previously trained model

# Check if dataset path and pretrained model exist
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")
if not os.path.exists(image_dir):
    raise FileNotFoundError(f"Images directory does not exist: {image_dir}")
if not os.path.exists(json_path):
    raise FileNotFoundError(f"Annotations file does not exist: {json_path}")
if not os.path.exists(pretrained_model_path):
    raise FileNotFoundError(f"Pretrained model not found: {pretrained_model_path}")

# Clear existing images and labels directories to avoid conflicts
output_base = dataset_path
for split in ['train', 'val', 'test']:
    images_split_dir = os.path.join(output_base, 'images', split)
    labels_split_dir = os.path.join(output_base, 'labels', split)
    if os.path.exists(images_split_dir):
        shutil.rmtree(images_split_dir)
    if os.path.exists(labels_split_dir):
        shutil.rmtree(labels_split_dir)
    os.makedirs(images_split_dir, exist_ok=True)
    os.makedirs(labels_split_dir, exist_ok=True)

# Load COCO annotations
with open(json_path, 'r') as f:
    coco_data = json.load(f)

# Map image IDs to filenames and sizes
image_id_to_filename = {img['id']: img['file_name'] for img in coco_data['images']}
image_id_to_size = {img['id']: (img['width'], img['height']) for img in coco_data['images']}

# Map category IDs to YOLO class IDs (0-based)
category_id_to_yolo = {cat['id']: idx for idx, cat in enumerate(coco_data['categories'])}
class_names = [cat['name'] for cat in coco_data['categories']]

# Verify class names
print("Class names from annotations:", class_names)
assert class_names == ['CreashBarrier', 'KerbPaint', 'MarkingPaint', 'RubberSpeedBreaker'], "Class names do not match expected classes."

# Split images
image_files = [img['file_name'] for img in coco_data['images']]
train_images, temp_images = train_test_split(image_files, test_size=0.2, random_state=42)
val_images, test_images = train_test_split(temp_images, test_size=0.5, random_state=42)

print(f"Train: {len(train_images)} images")
print(f"Validation: {len(val_images)} images")
print(f"Test: {len(test_images)} images")

# Convert COCO to YOLO format and move files
def convert_to_yolo(split, image_list):
    image_split_dir = os.path.join(output_base, 'images', split)
    label_split_dir = os.path.join(output_base, 'labels', split)

    for img_file in image_list:
        # Move image
        src_img = os.path.join(image_dir, img_file)
        dst_img = os.path.join(image_split_dir, img_file)
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
        else:
            print(f"Image not found: {src_img}")
            continue

        # Find image ID
        img_id = next(img['id'] for img in coco_data['images'] if img['file_name'] == img_file)
        img_width, img_height = image_id_to_size[img_id]

        # Get annotations for this image
        annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] == img_id]

        # Convert to YOLO format
        yolo_labels = []
        for ann in annotations:
            class_id = int(category_id_to_yolo[ann['category_id']])
            bbox = ann['bbox']  # [x_min, y_min, width, height]

            # Calculate YOLO coordinates
            x_center = (bbox[0] + bbox[2] / 2) / img_width
            y_center = (bbox[1] + bbox[3] / 2) / img_height
            width = bbox[2] / img_width
            height = bbox[3] / img_height

            # Clamp coordinates to [0.0, 1.0]
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            width = max(0.0, min(1.0, width))
            height = max(0.0, min(1.0, height))

            # Ensure x_max and y_max are in bounds
            x_max = x_center + width / 2
            y_max = y_center + height / 2
            if x_max > 1.0:
                width = (1.0 - x_center) * 2
            if y_max > 1.0:
                height = (1.0 - y_center) * 2
            if x_center - width / 2 < 0:
                width = x_center * 2
            if y_center - height / 2 < 0:
                height = y_center * 2

            # Ensure width and height are at least 0.001 to avoid zero-size boxes
            width = max(0.001, width)
            height = max(0.001, height)

            yolo_labels.append(f"{class_id} {x_center} {y_center} {width} {height}")

        # Generate label filename by removing any '_aug_X' suffix and replacing image extension with .txt
        base_name = os.path.splitext(img_file)[0]  # Get the filename without extension
        # Remove '_aug_X' suffix if present (e.g., 'image_aug_1' -> 'image')
        if '_aug_' in base_name:
            base_name = base_name.split('_aug_')[0]
        label_file = os.path.join(label_split_dir, f"{base_name}.txt")

        # Write YOLO label file
        with open(label_file, 'w') as f:
            f.write('\n'.join(yolo_labels))

# Process each split
convert_to_yolo('train', train_images)
convert_to_yolo('val', val_images)
convert_to_yolo('test', test_images)

# Create data.yaml for YOLOv8
data_yaml = f"""
train: {os.path.join(output_base, 'images', 'train')}
val: {os.path.join(output_base, 'images', 'val')}
test: {os.path.join(output_base, 'images', 'test')}

nc: {len(class_names)}
names: {class_names}
"""

data_yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

print("Dataset split and conversion to YOLO format completed.")
print(f"data.yaml created with class names: {class_names}")

# Define paths
splits = ['train', 'val', 'test']

# Define class names and updated oversampling factors for fine-tuning
oversample_factors = {
    'CreashBarrier': 4,  # Increased from 2 to focus on this class
    'KerbPaint': 3,      # Increased from 1 to focus on this class
    'MarkingPaint': 4,
    'RubberSpeedBreaker': 4
}

# Define enhanced augmentation pipeline
transform = A.Compose([
    A.Resize(height=640, width=640),
    A.HorizontalFlip(p=0.8),
    A.RandomRotate90(p=0.8),
    A.Rotate(limit=45, p=0.8),
    A.RandomScale(scale_limit=0.3, p=0.8),
    A.Affine(translate_percent=0.1, scale=(0.8, 1.2), rotate=(-45, 45), p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.8),
    A.GaussNoise(p=0.6),
    A.Blur(blur_limit=5, p=0.6),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.4, p=0.8),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Function to clamp bounding box coordinates
def clamp_bbox(bbox):
    center_x, center_y, width, height = bbox
    center_x = max(0.0, min(1.0, center_x))
    center_y = max(0.0, min(1.0, center_y))
    width = max(0.0, min(1.0, width))
    height = max(0.0, min(1.0, height))

    x_max = center_x + width / 2
    y_max = center_y + height / 2
    if x_max > 1.0:
        width = (1.0 - center_x) * 2
    if y_max > 1.0:
        height = (1.0 - center_y) * 2
    if center_x - width / 2 < 0:
        width = center_x * 2
    if center_y - height / 2 < 0:  # Fixed: y_center -> center_y
        height = center_y * 2      # Fixed: y_center -> center_y
    return [center_x, center_y, width, height]

# Preprocess and augment images with oversampling
def preprocess_images(image_dir, label_dir, output_image_dir, output_label_dir):
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)

    processed_count = 0
    image_files = os.listdir(image_dir)

    # Only process original images (exclude augmented images with '_aug_' in the name)
    image_files = [f for f in image_files if '_aug_' not in f]

    for img_file in image_files:
        img_path = os.path.join(image_dir, img_file)

        # Generate the base name for the label file (without '_aug_X' suffix)
        base_name = os.path.splitext(img_file)[0]
        label_path = os.path.join(label_dir, f"{base_name}.txt")

        # Read image and labels
        img = cv2.imread(img_path)
        if img is None:
            print(f"Skipping corrupted image: {img_file}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        try:
            with open(label_path, 'r') as f:
                labels = [line.strip().split() for line in f.readlines()]
        except Exception as e:
            print(f"Error reading label file {label_path}: {e}")
            continue

        if not labels:
            print(f"No annotations for {img_file}, copying without augmentation.")
            cv2.imwrite(os.path.join(output_image_dir, img_file), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            shutil.copy(label_path, os.path.join(output_label_dir, img_file.replace('.jpg', '.txt')))
            processed_count += 1
            continue

        class_labels = [int(label[0]) for label in labels]
        bboxes = [list(map(float, label[1:])) for label in labels]

        # Clamp bounding box coordinates
        bboxes = [clamp_bbox(bbox) for bbox in bboxes]

        # Determine oversampling factor based on the classes present
        max_factor = 1
        for class_id in class_labels:
            class_name = class_names[class_id]
            factor = oversample_factors[class_name]
            max_factor = max(max_factor, factor)

        # Apply augmentations for each oversampled instance
        for i in range(max_factor):
            try:
                augmented = transform(image=img, bboxes=bboxes, class_labels=class_labels)
                aug_img = augmented['image']
                aug_bboxes = augmented['bboxes']
                aug_labels = augmented['class_labels']
            except Exception as e:
                print(f"Error augmenting {img_file} (iteration {i+1}): {e}")
                continue

            # Save augmented image
            if i == 0:
                aug_img_file = img_file
            else:
                base, ext = os.path.splitext(img_file)
                aug_img_file = f"{base}_aug_{i}{ext}"

            aug_img = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(output_image_dir, aug_img_file), aug_img)

            # Save updated labels
            label_file = aug_img_file.replace('.jpg', '.txt')
            with open(os.path.join(output_label_dir, label_file), 'w') as f:
                for class_id, bbox in zip(aug_labels, aug_bboxes):
                    f.write(f"{class_id} {' '.join(map(str, bbox))}\n")

            processed_count += 1

    print(f"Processed {processed_count} images in {image_dir}")

# Preprocess all splits
for split in splits:
    print(f"\nPreprocessing {split} split...")
    image_dir = os.path.join(dataset_path, 'images', split)
    label_dir = os.path.join(dataset_path, 'labels', split)
    if not os.path.exists(image_dir) or not os.path.exists(label_dir):
        print(f"Directories not found: {image_dir}, {label_dir}")
        continue
    preprocess_images(
        image_dir,
        label_dir,
        image_dir,
        label_dir
    )

print("\nPreprocessing completed.")

# Load the previously trained YOLOv8m model
model = YOLO(pretrained_model_path)

# Fine-tune the model with adjusted hyperparameters
results = model.train(
    data=data_yaml_path,
    epochs=15,  # Fine-tune for 25 additional epochs
    imgsz=640,
    batch=32,  # Keep batch size; reduce to 16 if GPU memory is insufficient
    name='yolov8m_finetuned',
    project='/content/sample_data/Dataset/runs',
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=15,  # Reduce patience for fine-tuning
    save=True,
    val=True,
    exist_ok=True,
    cls=2.5,  # Increase to focus on classification
    box=8.5,  # Keep to maintain box precision
    lr0=0.0001,  # Lower learning rate for fine-tuning
    cos_lr=True,
    optimizer='AdamW',
    mixup=0.1,
)

# Evaluate the model on the test set
metrics = model.val(split='test')

# Print evaluation metrics
print("\nEvaluation Metrics on Test Set:")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50:95: {metrics.box.map:.4f}")
print("Per-class AP@50:")
for i, ap in enumerate(metrics.box.maps):
    print(f"{model.names[i]}: {ap:.4f}")

# Save the fine-tuned model
model.save('/content/sample_data/Dataset/yolov8m_finetuned.pt')

print("Fine-tuning and evaluation completed.")

Class names from annotations: ['CreashBarrier', 'KerbPaint', 'MarkingPaint', 'RubberSpeedBreaker']
Train: 2880 images
Validation: 360 images
Test: 360 images
Dataset split and conversion to YOLO format completed.
data.yaml created with class names: ['CreashBarrier', 'KerbPaint', 'MarkingPaint', 'RubberSpeedBreaker']

Preprocessing train split...
Processed 7881 images in /content/sample_data/Dataset/images/train

Preprocessing val split...
Processed 1009 images in /content/sample_data/Dataset/images/val

Preprocessing test split...
Processed 1010 images in /content/sample_data/Dataset/images/test

Preprocessing completed.
Ultralytics 8.3.140 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=8.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sample_data/Dataset/da

train: Scanning /content/sample_data/Dataset/labels/train... 7881 images, 742 backgrounds, 0 corrupt: 100%|██████████| 8611/8611 [00:04<00:00, 1749.96it/s]


train: New cache created: /content/sample_data/Dataset/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 440.9±97.0 MB/s, size: 262.6 KB)


val: Scanning /content/sample_data/Dataset/labels/val... 1009 images, 90 backgrounds, 0 corrupt: 100%|██████████| 1095/1095 [00:00<00:00, 1270.49it/s]


val: New cache created: /content/sample_data/Dataset/labels/val.cache
Plotting labels to /content/sample_data/Dataset/runs/yolov8m_finetuned/labels.jpg... 
optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/sample_data/Dataset/runs/yolov8m_finetuned
Starting training for 15 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all       1095       1435       0.88      0.841      0.871      0.654
         CreashBarrier        393        531      0.888      0.776      0.879      0.571
             KerbPaint        260        552      0.763      0.755      0.739      0.515
          MarkingPaint        352        352       0.99      0.991      0.995      0.876
Speed: 0.2ms preprocess, 10.4ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /content/sample_data/Dataset/runs/yolov8m_finetuned
Ultralytics 8.3.140 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 24.3±26.2 MB/s, size: 349.5 KB)


val: Scanning /content/sample_data/Dataset/labels/test... 1010 images, 87 backgrounds, 0 corrupt: 100%|██████████| 1094/1094 [00:04<00:00, 244.30it/s]

val: New cache created: /content/sample_data/Dataset/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:35<00:00,  1.01s/it]


                   all       1094       1381      0.875      0.837      0.876      0.662
         CreashBarrier        418        520       0.88      0.774      0.862      0.568
             KerbPaint        281        553      0.755      0.751       0.77      0.555
          MarkingPaint        308        308       0.99      0.986      0.995      0.863
Speed: 2.2ms preprocess, 23.7ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/sample_data/Dataset/runs/yolov8m_finetuned

Evaluation Metrics on Test Set:
mAP@50: 0.8759
mAP@50:95: 0.6621
Per-class AP@50:
CreashBarrier: 0.5678
KerbPaint: 0.5553
MarkingPaint: 0.8633
RubberSpeedBreaker: 0.6621
Fine-tuning and evaluation completed.
